# Scenario C

默认运行 C1、n=500、正式阶段，使用 12 个单线程 worker。改变条件时编辑设置单元。每次运行会自动跳过已提交记录。

In [ ]:
import importlib.util
from pathlib import Path
import sys, subprocess, os
roots = [Path.cwd(), *Path.cwd().parents, Path.home() / "Desktop" / "Rank_DML_Scenario_C"]
root = next(p for p in roots if (p / "代码" / "run.py").exists())
code = root / "代码"
dependencies = ["numpy", "scipy", "pandas", "matplotlib", "psutil", "tqdm", "threadpoolctl"]
missing = [name for name in dependencies if importlib.util.find_spec(name) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(code / "requirements.txt")], check=True)
print(sys.executable, root)

In [ ]:
STAGE = "main"  # "smoke", "pilot", "main"
CASE = "C1"      # "C1" or "C2"
N = 500
WORKERS = 12
REPS = None     # Set a smaller prefix for one cell, or None for its full target

In [ ]:
env = os.environ | {"PYTHONUTF8": "1", "PYTHONIOENCODING": "utf-8"}
command = [sys.executable, "-u", str(code / "run.py"), "--stage", STAGE,
           "--case", CASE, "--n", str(N), "--workers", str(WORKERS), "--resume"]
if REPS is not None:
    command.extend(["--reps", str(REPS)])
with subprocess.Popen(command, cwd=code, env=env, stdout=subprocess.PIPE,
                      stderr=subprocess.STDOUT, text=True, encoding="utf-8") as process:
    for line in process.stdout:
        print(line, end="", flush=True)
    if process.wait() != 0:
        raise RuntimeError(f"Experiment exited with status {process.returncode}")

全部正式条件完成后可重建汇总。以下命令只读取检查点。

In [ ]:
# subprocess.run([sys.executable, str(code / "check.py"), "main"], cwd=code, check=True)
# subprocess.run([sys.executable, str(code / "report.py")], cwd=code, check=True)